# TP-2: CleaningData - Applying OOP and Design Patterns

## Exercise 3: Add Decorators for Logging and Timing (Decorator Pattern)

**Objective:**
- Learn the **Decorator Pattern** to add reusable functionality (logging, timing).
- Understand how decorators support the **Open/Closed Principle**.

**Concept:** Decorators wrap functions to extend their behavior **without altering the original code**.

**Instructions:**
1. Create two decorators:
   - `@log_action` → logs when a method starts and finishes.
   - `@log_time` → calculates and prints execution time.
2. Apply them to methods in `CSVReader` or `DataCleaner`.

In [1]:
import os
import time
import functools
from abc import ABC, abstractmethod

import pandas as pd

### Step 1: Create the decorators
A decorator is a function that:
1. receives a function `func`,
2. defines a `wrapper` that adds extra behavior **before/after** calling `func`,
3. returns the `wrapper`.

`*args, **kwargs` let the wrapper accept any arguments, and `functools.wraps` keeps the original function's name and docstring.

In [2]:
def log_action(func):
    """Decorator: log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting '{func.__qualname__}'...")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished '{func.__qualname__}'")
        return result
    return wrapper


def log_time(func):
    """Decorator: measure and print the execution time of a function/method."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIME] '{func.__qualname__}' took {elapsed:.4f} seconds")
        return result
    return wrapper

### Quick test on a simple function

In [3]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.5)
    return a + b

print("Result:", slow_add(2, 3))
print("Function name is kept:", slow_add.__name__)

[LOG] Starting 'slow_add'...
[TIME] 'slow_add' took 0.5003 seconds
[LOG] Finished 'slow_add'
Result: 5
Function name is kept: slow_add


### Step 2: Apply the decorators to `CSVReader` and `DataCleaner`
We only add `@log_action` / `@log_time` above the methods. The body of each method is **exactly the same** as in Exercises 1 and 2.

In [4]:
class CSVReader:
    """Encapsulates reading and previewing a CSV file."""

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  # private: loaded DataFrame

    @log_action
    @log_time
    def read(self):
        """Load the CSV file into a pandas DataFrame and return it."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    @log_action
    def preview(self, n=5):
        """Print the top n rows of the data."""
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        """Read-only access to the loaded DataFrame."""
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

In [5]:
# Strategies from Exercise 2
class MissingValueStrategy(ABC):
    """Interface: every strategy must implement handle(df)."""

    @abstractmethod
    def handle(self, df):
        """Return a new DataFrame with missing values handled."""
        pass


class DropMissing(MissingValueStrategy):
    """Remove every row that has at least one missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing values in numeric columns with the column mean."""

    def handle(self, df):
        df = df.copy()
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
        return df


class FillMode(MissingValueStrategy):
    """Fill missing values in every column with its most frequent value."""

    def handle(self, df):
        df = df.copy()
        for col in df.columns:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0])
        return df

In [6]:
class DataCleaner:
    """Context class: applies whichever strategy it is given."""

    def __init__(self, strategy):
        self._strategy = strategy

    def set_strategy(self, strategy):
        """Switch to another strategy at runtime."""
        self._strategy = strategy

    @log_action
    @log_time
    def clean(self, df):
        print(f"Applying strategy: {type(self._strategy).__name__}")
        return self._strategy.handle(df)

### Test with `sample_data.csv`

In [7]:
reader = CSVReader("sample_data.csv")
df = reader.read()

[LOG] Starting 'CSVReader.read'...
[TIME] 'CSVReader.read' took 0.0017 seconds
[LOG] Finished 'CSVReader.read'


In [8]:
reader.preview(3)

[LOG] Starting 'CSVReader.preview'...
Top 3 rows of 'sample_data.csv':
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
[LOG] Finished 'CSVReader.preview'


In [9]:
cleaner = DataCleaner(FillMean())
cleaned = cleaner.clean(df)
cleaned

[LOG] Starting 'DataCleaner.clean'...
Applying strategy: FillMean
[TIME] 'DataCleaner.clean' took 0.0020 seconds
[LOG] Finished 'DataCleaner.clean'


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


### Summary
- `@log_action` prints when a method **starts** and **finishes**.
- `@log_time` measures how long a method takes with `time.perf_counter()`.
- **Stacking**: `@log_action` on top of `@log_time` means `log_action(log_time(read))`, so the log wraps the timer.
- **Reusable**: the same decorators work on any function or method (`slow_add`, `read`, `clean`, ...).
- **Open/Closed Principle**: classes are *open for extension* (new logging/timing behavior) but *closed for modification* (the method code did not change).